In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
class LocalFileAdapter:
    """Adapter for local file paths to work with ChatIngestor."""

    def __init__(self, file_path: str):
        self.path = Path(file_path)
        self.name = self.path.name

    def getbuffer(self) -> bytes:
        return self.path.read_bytes()

In [2]:
import pandas as pd

# Path to your semiconductor report text file
text_file_path = r"C:\Users\Rajesh\Downloads\global_semiconductor_industry_report.txt"

# Read the txt file (optional, just to verify loading)
with open(text_file_path, "r", encoding="utf-8") as file:
    report_text = file.read()

print("Document loaded successfully!")
print(f"Document length: {len(report_text)} characters")


# QA
inputs = [
    "What was the approximate global semiconductor industry revenue in 2023?",
    
    "Which company is the sole commercial supplier of EUV lithography systems?",
    
    "Why is Taiwan considered strategically important in the global semiconductor industry?",
    
    "How did the AI boom impact NVIDIA's financial growth between fiscal years 2023 and 2024?",
    
    "What are the main challenges currently facing the semiconductor industry?",
]

outputs = [
    "The global semiconductor industry generated approximately $527 billion in revenues in 2023.",
    
    "ASML Holding is the sole commercial supplier of EUV lithography systems used for advanced chip manufacturing.",
    
    "Taiwan is strategically important because it produces approximately 92 percent of the world's most advanced logic chips, with TSMC dominating advanced semiconductor manufacturing.",
    
    "NVIDIA's revenues increased dramatically from approximately $26.9 billion in fiscal year 2023 to $60.9 billion in fiscal year 2024 due to explosive demand for AI accelerators and data center GPUs.",
    
    "The semiconductor industry faces challenges including rising fabrication costs, power consumption, thermal management issues, physical scaling limitations, water usage concerns, supply chain fragility, and a global shortage of semiconductor engineers.",
]

# Dataset
qa_pairs = [{"question": q, "answer": a} for q, a in zip(inputs, outputs)]

df = pd.DataFrame(qa_pairs)

# Save QA pairs as CSV
csv_path = r"C:\Users\Rajesh\Desktop\LLMOPS\data\goldens.csv"

df.to_csv(csv_path, index=False)

print("\nQA Dataset Created Successfully!")
print(df)

Document loaded successfully!
Document length: 67139 characters

QA Dataset Created Successfully!
                                            question  \
0  What was the approximate global semiconductor ...   
1  Which company is the sole commercial supplier ...   
2  Why is Taiwan considered strategically importa...   
3  How did the AI boom impact NVIDIA's financial ...   
4  What are the main challenges currently facing ...   

                                              answer  
0  The global semiconductor industry generated ap...  
1  ASML Holding is the sole commercial supplier o...  
2  Taiwan is strategically important because it p...  
3  NVIDIA's revenues increased dramatically from ...  
4  The semiconductor industry faces challenges in...  


In [5]:
from langsmith import Client

client = Client()
dataset_name = "LLMOPS"

# Store
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Input and expected output pairs for LLMOPS",
)
client.create_examples(
    inputs=[{"question": q} for q in inputs],
    outputs=[{"answer": a} for a in outputs],
    dataset_id=dataset.id,
)

{'example_ids': ['74076054-aa50-420a-8318-70fbf7b04d52',
  '2522dabf-e6f3-45e1-b59d-684707c85553',
  'c40fdd99-13c1-4c5d-9d94-3f668fd2b1f5',
  '514ec0a9-2785-4ef1-a829-aab970f764ea',
  '9003b539-1fbd-4b7f-95de-746fd2d9f69c'],
 'count': 5,
 'as_of': '2026-05-19T22:28:42.648993577Z'}

In [7]:
import sys
sys.path.append(r"C:\Users\Rajesh\Desktop\LLMOPS")

from pathlib import Path
from multi_doc_chat.src.document_ingestion.data_ingestion import ChatIngestor
from multi_doc_chat.src.document_chat.retrieval import ConversationalRAG

import os

In [9]:
# Path
data_path = r"C:\Users\Rajesh\Downloads\global_semiconductor_industry_report.txt"

# Create adapter
file_adapter = LocalFileAdapter(data_path)

# Build retriever once
ingestor = ChatIngestor(
    temp_base="data",
    faiss_base="faiss_index",
    use_session_dirs=True
)

ingestor.built_retriver(
    uploaded_files=[file_adapter],
    chunk_size=2000,
    chunk_overlap=100,
    k=5
)

# Load FAISS once
session_id = ingestor.session_id
index_path = f"faiss_index/{session_id}"

rag = ConversationalRAG(session_id=session_id)

rag.load_retriever_from_faiss(
    index_path=index_path,
    k=5,
    index_name=os.getenv("FAISS_INDEX_NAME", "index")
)

{"timestamp": "2026-05-21T01:01:24.652657Z", "level": "info", "event": "Running in LOCAL mode: .env loaded"}
{"timestamp": "2026-05-21T01:01:24.661651Z", "level": "info", "event": "Loaded OPENROUTER_API_KEY from individual env var"}
{"keys": {"OPENROUTER_API_KEY": "sk-or-..."}, "timestamp": "2026-05-21T01:01:24.663634Z", "level": "info", "event": "API keys loaded"}
{"config_keys": ["llm", "embedding_model", "retriever"], "timestamp": "2026-05-21T01:01:24.683073Z", "level": "info", "event": "YAML config loaded"}
{"session_id": "session_20260521_063124_fd703870", "temp_dir": "data\\session_20260521_063124_fd703870", "faiss_dir": "faiss_index\\session_20260521_063124_fd703870", "sessionized": true, "timestamp": "2026-05-21T01:01:24.693702Z", "level": "info", "event": "ChatIngestor initialized"}
{"uploaded": "global_semiconductor_industry_report.txt", "saved_as": "data\\session_20260521_063124_fd703870\\d9323f34.txt", "timestamp": "2026-05-21T01:01:24.704705Z", "level": "info", "event": "F

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026367C46010>, search_type='mmr', search_kwargs={'k': 5, 'fetch_k': 20, 'lambda_mult': 0.5})

In [10]:
def answer_ai_report_question(inputs: dict) -> dict:

    try:

        question = inputs.get("question", "")

        if not question:
            return {"answer": "No question provided"}

        answer = rag.invoke(
            question,
            chat_history=[]
        )

        return {"answer": answer}

    except Exception as e:

        return {"answer": f"Error: {str(e)}"}

In [11]:
# Test the function with a sample question

test_input = {
    "question": "What was the approximate global semiconductor industry revenue in 2023?"
}

result = answer_ai_report_question(test_input)

print("Question:", test_input["question"])
print("\nAnswer:", result["answer"])

HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20260521_063124_fd703870", "user_input": "What was the approximate global semiconductor industry revenue in 2023?", "answer_preview": "The global semiconductor industry revenue in 2023 was approximately $527 billion.", "timestamp": "2026-05-21T01:01:46.527585Z", "level": "info", "event": "Chain invoked successfully"}


Question: What was the approximate global semiconductor industry revenue in 2023?

Answer: The global semiconductor industry revenue in 2023 was approximately $527 billion.


In [12]:
from langsmith.evaluation import evaluate

In [13]:
# Example: Test with all golden questions
print("Testing all questions from the dataset:\n")
for i, q in enumerate(inputs, 1):
    test_input = {"question": q}
    result = answer_ai_report_question(test_input)
    print(f"Q{i}: {q}")
    print(f"A{i}: {result['answer']}\n")
    print("-" * 80 + "\n")

Testing all questions from the dataset:



HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20260521_063124_fd703870", "user_input": "What was the approximate global semiconductor industry revenue in 2023?", "answer_preview": "The global semiconductor industry revenue in 2023 was approximately $527 billion.", "timestamp": "2026-05-21T01:01:58.024215Z", "level": "info", "event": "Chain invoked successfully"}


Q1: What was the approximate global semiconductor industry revenue in 2023?
A1: The global semiconductor industry revenue in 2023 was approximately $527 billion.

--------------------------------------------------------------------------------



HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20260521_063124_fd703870", "user_input": "Which company is the sole commercial supplier of EUV lithography systems?", "answer_preview": "ASML Holding is the sole commercial supplier of EUV lithography systems.", "timestamp": "2026-05-21T01:02:00.080260Z", "level": "info", "event": "Chain invoked successfully"}


Q2: Which company is the sole commercial supplier of EUV lithography systems?
A2: ASML Holding is the sole commercial supplier of EUV lithography systems.

--------------------------------------------------------------------------------



...

Q3: Why is Taiwan considered strategically important in the global semiconductor industry?
A3: Taiwan is considered strategically important in the global semiconductor industry due to its concentration of leading-edge production, particularly at TSMC's facilities, which produce approximately 92 percent of the world's most advanced logic chips. The ecosystem benefits from government investment, a skilled workforce, specialized suppliers, and co-location of the entire value chain, making it a critical hub for semiconductor manufacturing.

--------------------------------------------------------------------------------



HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20260521_063124_fd703870", "user_input": "How did the AI boom impact NVIDIA's financial growth between fiscal years 2023 and 2024?", "answer_preview": "The AI boom significantly impacted NVIDIA's financial growth between fiscal years 2023 and 2024. The company's data center segment, including AI accel", "timestamp": "2026-05-21T01:02:05.787556Z", "level": "info", "event": "Chain invoked successfully"}


Q4: How did the AI boom impact NVIDIA's financial growth between fiscal years 2023 and 2024?
A4: The AI boom significantly impacted NVIDIA's financial growth between fiscal years 2023 and 2024. The company's data center segment, including AI accelerator products, grew from $15 billion to over $47 billion during this period, representing an extraordinary year-over-year growth rate of more than 200 percent.

--------------------------------------------------------------------------------



HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20260521_063124_fd703870", "user_input": "What are the main challenges currently facing the semiconductor industry?", "answer_preview": "The main challenges currently facing the semiconductor industry include extreme water consumption, global talent shortage for semiconductor engineers,", "timestamp": "2026-05-21T01:02:07.838782Z", "level": "info", "event": "Chain invoked successfully"}


Q5: What are the main challenges currently facing the semiconductor industry?
A5: The main challenges currently facing the semiconductor industry include extreme water consumption, global talent shortage for semiconductor engineers, environmental concerns related to manufacturing processes, and the high capital intensity required for constructing fabrication facilities.

--------------------------------------------------------------------------------



In [14]:
from langsmith.evaluation import evaluate
from langchain.evaluation import load_evaluator
from langchain_openai import ChatOpenAI

# Evaluator LLM
eval_llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="openai/gpt-3.5-turbo",
    temperature=0
)

# Load evaluator
evaluator = load_evaluator(
    "labeled_criteria",
    criteria="correctness",
    llm=eval_llm
)

# Dataset name
dataset_name = "LLMOPS"

# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=[evaluator],

    experiment_prefix="test-semiconductor-rag",

    metadata={
        "variant": "RAG with FAISS and Semiconductor Industry Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("Evaluation completed!")

View the evaluation results for experiment: 'test-semiconductor-rag-792d1a0f' at:
https://smith.langchain.com/o/fdd801ea-9f2f-4e94-b47a-93641a50baef/datasets/b8a0a0fe-0013-4f7d-971c-8e0daf98c150/compare?selectedSessions=c84a2b83-dc2f-441f-8bb0-d8d08b008296




0it [00:00, ?it/s]

HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20260521_063124_fd703870", "user_input": "What are the main challenges currently facing the semiconductor industry?", "answer_preview": "The main challenges currently facing the semiconductor industry include extreme water consumption, global talent shortage for semiconductor engineers,", "timestamp": "2026-05-21T01:02:19.798490Z", "level": "info", "event": "Chain invoked successfully"}
c:\Users\Rajesh\Desktop\LLMOPS\.venv\Lib\site-packages\langsmith\evaluation\evaluator.py:758: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return func(*args, **kwargs)
Error running evaluator <DynamicRunEvaluator wrapper> on run 019e480d-ffb7-7f00-bf31-0c5c2fa1ef5b: ValueError("Missing some input keys: {'reference

Evaluation completed!


In [15]:
from langsmith.evaluation import evaluate
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
import os


# Evaluator LLM
eval_llm = ChatOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    model="openai/gpt-3.5-turbo",
    temperature=0
)


# Custom evaluator function
def correctness_evaluator(run, example):

    question = example.inputs["question"]

    reference_answer = example.outputs["answer"]

    generated_answer = run.outputs["answer"]

    prompt = f"""
    You are an expert evaluator for RAG systems.

    Question:
    {question}

    Ground Truth Answer:
    {reference_answer}

    Generated Answer:
    {generated_answer}

    Evaluate whether the generated answer is factually correct and semantically equivalent to the ground truth answer.

    Respond ONLY with:
    CORRECT
    or
    INCORRECT
    """

    result = eval_llm.invoke(prompt).content.strip()

    score = 1 if "CORRECT" in result else 0

    return {
        "key": "correctness",
        "score": score,
        "comment": result,
    }


# Dataset name
dataset_name = "LLMOPS"


# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=[correctness_evaluator],

    experiment_prefix="test-semiconductor-rag",

    metadata={
        "variant": "RAG with FAISS and Semiconductor Industry Report",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 3,
    },
)

print("Evaluation completed!")

View the evaluation results for experiment: 'test-semiconductor-rag-5daef6ef' at:
https://smith.langchain.com/o/fdd801ea-9f2f-4e94-b47a-93641a50baef/datasets/b8a0a0fe-0013-4f7d-971c-8e0daf98c150/compare?selectedSessions=18d86260-0690-4322-ac56-17801b54b00c




0it [00:00, ?it/s]

HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20260521_063124_fd703870", "user_input": "What are the main challenges currently facing the semiconductor industry?", "answer_preview": "The main challenges currently facing the semiconductor industry include extreme water consumption, global talent shortage for semiconductor engineers,", "timestamp": "2026-05-21T01:04:39.102124Z", "level": "info", "event": "Chain invoked successfully"}
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20260521_063124_fd703870", "user_input": "Which company is the sole commercial supplier of EUV lithography systems?", "answer_preview": "ASML Holdi

Evaluation completed!


In [16]:
from langsmith.schemas import Run, Example
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os


def correctness_evaluator(run: Run, example: Example) -> dict:
    """
    Custom LLM-as-a-Judge evaluator for correctness.

    Correctness means how well the actual model output matches
    the reference output in terms of factual accuracy,
    semantic meaning, and coverage.
    """

    # Extract outputs
    actual_output = run.outputs.get("answer", "")
    expected_output = example.outputs.get("answer", "")
    input_question = example.inputs.get("question", "")

    # Evaluation prompt
    eval_prompt = ChatPromptTemplate.from_messages([
        (
            "system",
            """You are an expert evaluator for RAG systems.

Your task is to judge whether the generated answer matches
the reference answer semantically and factually.

Evaluation Rules:
- If the generated answer conveys the same meaning as the reference answer, mark it CORRECT.
- Ignore wording differences and paraphrasing.
- Focus on factual correctness and semantic alignment.
- If important facts are missing or incorrect, mark it INCORRECT.
- Do not judge writing style or formatting."""
        ),

        (
            "human",
            """Question:
{input}

Reference Answer:
{expected_output}

Generated Answer:
{actual_output}

Evaluate the generated answer.

Respond EXACTLY in this format:

Reasoning: <brief reasoning>

Verdict: CORRECT
or
Verdict: INCORRECT
"""
        )
    ])

    # OpenRouter Judge LLM
    llm = ChatOpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.getenv("OPENROUTER_API_KEY"),
        model="nousresearch/hermes-3-llama-3.1-405b",
        temperature=0
    )

    # Create evaluation chain
    chain = eval_prompt | llm

    try:
        # Invoke evaluator
        response = chain.invoke({
            "input": input_question,
            "expected_output": expected_output,
            "actual_output": actual_output
        })

        response_text = response.content

        # Parse response
        reasoning = ""
        verdict = ""

        for line in response_text.split("\n"):

            if line.startswith("Reasoning:"):
                reasoning = line.replace("Reasoning:", "").strip()

            elif line.startswith("Verdict:"):
                verdict = line.replace("Verdict:", "").strip()

        # Convert verdict to score
        score = 1 if "CORRECT" in verdict.upper() else 0
        
        return {
            "key": "correctness",
            "score": score,
            "comment": f"{reasoning} | Verdict: {verdict}"
            }

    except Exception as e:

        return {
            "key": "correctness",
            "score": 0,
            "comment": f"Evaluation Error: {str(e)}"
            }

In [17]:
# Run evaluation with the custom correctness evaluator

from langsmith.evaluation import evaluate

# Define evaluators
evaluators = [correctness_evaluator]

dataset_name = "LLMOPS"

# Run evaluation
experiment_results = evaluate(
    answer_ai_report_question,
    data=dataset_name,
    evaluators=evaluators,

    experiment_prefix="semiconductor-rag-correctness-eval",

    description="Evaluating Semiconductor Industry RAG system using LLM-as-a-Judge correctness evaluation",

    metadata={
        "variant": "RAG with FAISS and Semiconductor Industry Report",
        "evaluator": "custom_correctness_llm_judge",
        "model": "nousresearch/hermes-3-llama-3.1-405b",
        "chunk_size": 1000,
        "chunk_overlap": 200,
        "k": 5,
    },
)

print("\nEvaluation completed! Check LangSmith for detailed results.")

View the evaluation results for experiment: 'semiconductor-rag-correctness-eval-ffef90db' at:
https://smith.langchain.com/o/fdd801ea-9f2f-4e94-b47a-93641a50baef/datasets/b8a0a0fe-0013-4f7d-971c-8e0daf98c150/compare?selectedSessions=ab4274d2-d657-4c76-a0dd-18285c17c759




0it [00:00, ?it/s]

HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20260521_063124_fd703870", "user_input": "What are the main challenges currently facing the semiconductor industry?", "answer_preview": "The main challenges currently facing the semiconductor industry include extreme water consumption, global talent shortage for semiconductor engineers,", "timestamp": "2026-05-21T01:06:36.463523Z", "level": "info", "event": "Chain invoked successfully"}
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
HTTP Request: POST https://openrouter.ai/api/v1/chat/completions "HTTP/1.1 200 OK"
{"session_id": "session_20260521_063124_fd703870", "user_input": "Which company is the sole commercial supplier of EUV lithography systems?", "answer_preview": "ASML Holdi


Evaluation completed! Check LangSmith for detailed results.
